In [ ]:
# Import more libraries
from sklearn.linear_model import LinearRegression
from scipy.stats import ttest_ind
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd


In [ ]:
# Open the dataset
df = pd.read_csv("/content/NSCLCMultiOmicsDataProcessed_reordered.csv")

# View the first 5 rows
print(df.head())

In [ ]:
print("Presenting the DataFrame:")
print("")
print("Variables: Four types of omics.")
print("Number of RNAseq variables:",
      len([match for match in df.columns if match.startswith("rs")]))
print("Number of Copy Number variables:",
      len([match for match in df.columns if match.startswith("cn")]))
print("Number of Mutation variables:",
      len([match for match in df.columns if match.startswith("mu")]))
print("Number of Protein variables:",
      len([match for match in df.columns if match.startswith("pp")]))

print("")
print("Outcomes: Outcomes.")
print("NSCLC Metastasis Status:",df["metastasis_status"].unique())


In [ ]:
# Isolate columns with information about this gene
df_myh11 = df[[col for col in df.columns if "MYH11" in col]]
print("MYH11 Variables:",list(df_myh11.columns))

In [ ]:
for col in df_myh11.columns:

    # perform the t-test
    print("T-Test on relationship between %s and patient survival"%col)
    print("Mean in Non-Metastasis:",df.loc[df["metastasis_status"]==0,col].mean())
    print("Mean in Metastasis:",df.loc[df["metastasis_status"]==1,col].mean())
    t_stat, p_val = ttest_ind(df.loc[df["metastasis_status"]==0,col],
                              df.loc[df["metastasis_status"]==1,col])

    # print the results
    print('T-statistic: ', t_stat)
    print('P-value: ', p_val)
    print("")

In [ ]:
# calculate effect of PP on status
model = LinearRegression(fit_intercept=True)
model.fit(df[["pp_MYH11"]], df["metastasis_status"])
cn_on_status = model.coef_[0]


print("Total Effect of PP on Status:",cn_on_status)

In [ ]:
outcomes = df[["metastasis_status"]]
df_mu = df[[col for col in df if col.startswith('mu')]]
df_cn = df[[col for col in df if col.startswith('cn')]]
df_rs = df[[col for col in df if col.startswith('rs')]]
df_pp = df[[col for col in df if col.startswith('pp')]]

In [ ]:
# Calculate Pearson's correlation coefficients between copy number and RNA-seq data
correlation_matrix = np.corrcoef(df_cn, df_rs, rowvar=False)[:df_cn.shape[1], df_cn.shape[1]:]

# Create a dataframe from the correlation matrix
correlation_df = pd.DataFrame(correlation_matrix, index=df_cn.columns, columns=df_rs.columns)

In [ ]:
# Plot the heatmap
plt.figure(figsize=(20, 20))
sns.clustermap(correlation_df, cmap="coolwarm")

# Customize plot appearance
plt.title("Pairwise Pearson's Correlations between Copy Number and RNA-seq Data")

# Show the plot
plt.show()

In [ ]:
# Combine the dataframes
combined_data = pd.concat([df_cn,df_rs,df_mu,df_pp], axis=1)

# Standardize the combined data
standardized_data = (combined_data - combined_data.mean()) / combined_data.std()

# Apply PCA to reduce dimensionality
n_components = 2
pca = PCA(n_components=n_components)
reduced_data = pca.fit_transform(standardized_data)

# Plot the reduced data
plt.figure(figsize=(10, 10))
plt.scatter(reduced_data[:, 0], reduced_data[:, 1], s=50)

# Customize plot appearance
plt.title("PCA of Combined Copy Number and RNA-seq Data")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

# Show the plot
plt.show()

In [ ]:
from sklearn.cluster import KMeans

# Apply PCA to reduce dimensionality
n_components = 2
pca = PCA(n_components=n_components)
reduced_data = pca.fit_transform(standardized_data)

# Apply KMeans clustering
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_assignments = kmeans.fit_predict(standardized_data)

# Generate set of colors for clusters
colors = []
for ii in range(n_clusters):
    colors.append("#{:06x}".format(np.random.randint(0, 0xFFFFFF)))

# Plot the reduced data with cluster colors
plt.figure(figsize=(10, 10))
for cluster in range(n_clusters):
    cluster_points = reduced_data[cluster_assignments == cluster]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], s=50, label=f"Cluster {cluster + 1}",color=colors[cluster])

# Customize plot appearance
plt.title("PCA of Combined Copy Number and RNA-seq Data with KMeans Clustering")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()

# Show the plot
plt.show()

In [ ]:
# Let's overlay survival rate on top of these clusters?

survived = (outcomes["metastasis_status"]).to_numpy()


# Define marker shapes for the survived array
marker_shapes = {0: '.', 1: 'o'}

# Plot the reduced data with cluster colors and different shapes for survived values
plt.figure(figsize=(10, 10))
for cluster in range(n_clusters):
    color = colors[cluster]
    for survived_value, marker_shape in marker_shapes.items():
        cluster_points = reduced_data[(cluster_assignments == cluster) & (survived == survived_value)]
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], s=50, marker=marker_shape,
                    label=f"Cluster {cluster + 1}, Metastasis: {survived_value}",color=color)


# Customize plot appearance
plt.title("PCA of Combined Copy Number and RNA-seq Data with KMeans Clustering")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()

# Show the plot
plt.show()

In [ ]:
# Proportions of survival in each cluster
cluster_survival = []
for cluster in range(n_clusters):
    cluster_survival.append(np.sum((cluster_assignments==cluster)&(survived==1))/np.sum(cluster_assignments==cluster))

labels = []
for ii in range(n_clusters):
    labels.append("Cluster %i"%(ii+1))

# create a bar plot
plt.bar(labels, cluster_survival,color=colors)
# plot mean
plt.plot([-1,4],[np.mean(survived),np.mean(survived)],'--',color='gray')

# add a title to the plot
plt.title('Proportion of survived members in each cluster')

# add labels to the x and y axes
plt.xlabel('X-axis label')
plt.ylabel('Y-axis label')

# display the plot
plt.show()

In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.stats import kendalltau

# Define the path to the processed data file
data_file_path = "/content/NSCLCMultiOmicsDataProcessed_reordered.csv"

# Read the data into a DataFrame
df = pd.read_csv(data_file_path)

# Extract individual omics dataframes
df_cn = df[[col for col in df.columns if col.startswith('cn')]]
df_rs = df[[col for col in df.columns if col.startswith('rs')]]
df_mu = df[[col for col in df.columns if col.startswith('mu')]]
df_pp = df[[col for col in df.columns if col.startswith('pp')]]

# Function for t-SNE analysis and Kendall correlation visualization
def perform_tsne_and_plot(data, omics_type):
    # Standardize the data
    standardized_data = (data - data.mean()) / data.std()

    # Apply t-SNE to reduce dimensionality
    tsne = TSNE(n_components=2, random_state=42)
    reduced_data = tsne.fit_transform(standardized_data)

    # Calculate Kendall correlation between t-SNE components and metastasis status
    kendall_corr, _ = kendalltau(reduced_data[:, 0], reduced_data[:, 1])

    # Plot the reduced data
    plt.figure(figsize=(10, 10))

    # Specify colors for Metastatic and Non-Metastatic
    colors = ['purple' if status == 0 else 'yellow' for status in df["metastasis_status"]]
    plt.scatter(reduced_data[:, 0], reduced_data[:, 1], s=50, c=colors, cmap="viridis")

    # Customize plot appearance with correlation value
    plt.title(f"t-SNE of {omics_type} Data with Metastasis Status\nKendall Correlation: {kendall_corr:.2f}")
    plt.xlabel("t-SNE Component 1")
    plt.ylabel("t-SNE Component 2")

    # Create a legend for the colors
    legend_labels = {0: 'Non-Metastatic', 1: 'Metastatic'}
    legend_handles = [plt.Line2D([0], [0], marker='o', color=color, label=label,
                                 markerfacecolor=color, markersize=10) for color, label in zip(['purple', 'yellow'], legend_labels.values())]
    plt.legend(handles=legend_handles, title="Metastasis Status")

    # Show the plot
    plt.show()

# Perform t-SNE and plot for each omics type
perform_tsne_and_plot(df_cn, "Copy Number Variation")
perform_tsne_and_plot(df_rs, "RNA Sequencing")
perform_tsne_and_plot(df_mu, "Somatic Mutations")
perform_tsne_and_plot(df_pp, "Reverse Phospho-Protein Arrays")


In [ ]:
# Print information about the DataFrame
print(df.info())



Model 1

In [ ]:
#Model 1


from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GaussianNoise, Lambda
from tensorflow.keras import backend as K

# Function to add Gaussian noise to the input for denoising autoencoder
def add_noise(x, noise_factor=0.5):
    noise = K.random_normal(shape=K.shape(x), mean=0.0, stddev=noise_factor)
    return x + noise

# Extract the features (rs_ data) and target variable
X_rs = df[[col for col in df.columns if col.startswith('rs_')]]
y = df["metastasis_status"]

# Apply SMOTE to the entire dataset
smote = SMOTE(sampling_strategy=0.6, random_state=42)
X_augmented, y_augmented = smote.fit_resample(X_rs, y)

# Split the augmented data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_augmented, y_augmented, test_size=0.2, random_state=42)

# Standardize and apply PCA to reduce dimensionality
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

# Add Gaussian noise for denoising autoencoder
noise_factor = 50
X_train_noisy = add_noise(X_train_std, noise_factor)
X_test_noisy = add_noise(X_test_std, noise_factor)

# Create and train a denoising autoencoder
input_layer = Input(shape=(X_train_std.shape[1],))
noisy_input = GaussianNoise(stddev=noise_factor)(input_layer)
encoded = Dense(64, activation='relu')(noisy_input)
encoded = Dense(32, activation='relu')(encoded)
decoded = Dense(64, activation='relu')(encoded)
decoded = Dense(X_train_std.shape[1], activation='linear')(decoded)

autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mean_squared_error')
autoencoder.fit(X_train_noisy, X_train_std, epochs=50, batch_size=32, shuffle=True, validation_data=(X_test_noisy, X_test_std))

# Extract the top 100 features from the encoder part of the autoencoder
encoder = Model(input_layer, encoded)
X_train_encoded = encoder.predict(X_train_std)
X_test_encoded = encoder.predict(X_test_std)

# Extract the top 100 most relevant features
# Ensure that the number of selected features is less than or equal to the original feature count
# Extract the top 100 features from the encoder part of the autoencoder
top_features_idx = X_train_encoded.var(axis=0).argsort()[:max(100, X_train_encoded.shape[1])]


# Train a random forest model on the top 100 features
forest_model_top_500_features = RandomForestClassifier(random_state=42)
forest_model_top_500_features.fit(X_train_std[:, top_features_idx], y_train)

# Make predictions on the test set using the top 100 features
y_pred_forest_top_500_features = forest_model_top_500_features.predict(X_test_std[:, top_features_idx])

# Evaluate the random forest model with the top 100 features
accuracy_forest_top_500_features = accuracy_score(y_test, y_pred_forest_top_500_features)

print("Random Forest Model with Top 100 Features:")
print("Accuracy:", accuracy_forest_top_500_features)
print("Classification Report:\n", classification_report(y_test, y_pred_forest_top_500_features))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_forest_top_500_features))

# Display the top 30 most relevant features without the rs_ prefix
top_30_features_idx = top_features_idx[:30]
top_30_features = X_train.columns[top_30_features_idx]
top_30_features_without_prefix = [col.split('_')[1] for col in top_30_features]
print("\nTop 30 Features without rs_ prefix:", top_30_features_without_prefix)
from sklearn.metrics import roc_auc_score, roc_curve

# Compute probabilities of the positive class
y_prob_forest_top_500_features = forest_model_top_500_features.predict_proba(X_test_std[:, top_features_idx])[:, 1]

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob_forest_top_500_features)

# Compute AUC score
auc_score = roc_auc_score(y_test, y_prob_forest_top_500_features)

print("AUC Score:", auc_score)

# Plot ROC curve (optional)
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % auc_score)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()


In [ ]:
from sklearn.decomposition import PCA
from tensorflow.keras.layers import GaussianNoise

# Apply PCA to reduce dimensionality
pca = PCA(n_components=100)  # Assuming you want to keep 100 principal components
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)

# Add Gaussian noise to the input for denoising autoencoder
noise_factor = 0.5  # Adjust as needed
X_train_noisy = add_noise(X_train_pca, noise_factor)
X_test_noisy = add_noise(X_test_pca, noise_factor)

# Define the Deep Multi-View Learning model
def create_multi_view_model(input_shape):
    input_layer = Input(shape=input_shape)
    noisy_input = GaussianNoise(stddev=noise_factor)(input_layer)  # Add Gaussian noise to the input
    hidden_layer1 = Dense(64, activation='relu')(noisy_input)
    hidden_layer2 = Dense(32, activation='relu')(hidden_layer1)
    output_layer = Dense(1, activation='sigmoid')(hidden_layer2)
    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Create the Deep Multi-View Learning model
input_shape = X_train_pca.shape[1]
multi_view_model = create_multi_view_model(input_shape)

# Train the model
multi_view_model.fit(X_train_noisy, y_train, epochs=50, batch_size=32, validation_data=(X_test_noisy, y_test))

# Evaluate the model
y_pred = multi_view_model.predict(X_test_noisy)
y_pred_binary = (y_pred > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred_binary)
classification_rep = classification_report(y_test, y_pred_binary)
conf_matrix = confusion_matrix(y_test, y_pred_binary)

print("Deep Multi-View Learning Model with Denoising Autoencoder and PCA:")
print("Accuracy:", accuracy)
print("Classification Report:\n", classification_rep)
print("Confusion Matrix:\n", conf_matrix)

from sklearn.metrics import roc_auc_score

# Evaluate the model
y_pred = multi_view_model.predict(X_test_noisy)
y_pred_binary = (y_pred > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred_binary)
classification_rep = classification_report(y_test, y_pred_binary)
conf_matrix = confusion_matrix(y_test, y_pred_binary)
auc = roc_auc_score(y_test, y_pred)  # Calculate AUC

print("Deep Multi-View Learning Model with Denoising Autoencoder and PCA:")
print("Accuracy:", accuracy)
print("AUC:", auc)
print("Classification Report:\n", classification_rep)
print("Confusion Matrix:\n", conf_matrix)

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Make predictions on the test set using the top 500 features
y_pred_forest_top_500_features = forest_model_top_500_features.predict(X_test_std[:, top_features_idx])

# Calculate Cohen's Kappa
kappa_forest_top_500_features = cohen_kappa_score(y_test, y_pred_forest_top_500_features)

# Display Cohen's Kappa
print("\nCohen's Kappa for Random Forest Model with Top 500 Features:")
print(kappa_forest_top_500_features)


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Get predicted probabilities for the positive class
y_prob_forest_top_500_features = forest_model_top_500_features.predict_proba(X_test_std[:, top_features_idx])[:, 1]

# Calculate ROC curve
fpr_rs, tpr_rs, thresholds_rs = roc_curve(y_test, y_prob_forest_top_500_features)

# Calculate AUC (Area Under the Curve)
roc_auc_rs = auc(fpr_rs, tpr_rs)
final_roc_curve_cn = {
    'fpr': fpr_rs,
    'tpr': tpr_rs,
    'thresholds': thresholds_rs,
    'roc_auc': roc_auc_rs
}
# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_rs, tpr_rs, color='darkorange', lw=2, label='ROC curve (AUC = {:.2f})'.format(roc_auc_rs))
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()


In [ ]:
# Save the denoising autoencoder
autoencoder.save("denoising_autoencoder_model.h5")

# Save the random forest model with top 500 features
import joblib
joblib.dump(forest_model_top_500_features, "random_forest_model_top_500_features.joblib")


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix

# Load the new dataset
new_data = pd.read_csv("/content/NSCLCMultiOmicsDataProcessed_reordered.csv")

# Extract the features (rs_ data)
X_new = new_data[[col for col in new_data.columns if col.startswith('rs_')]]

# Use the top 100 features for prediction (adjust the number as needed)
X_new_top_100 = X_new.iloc[:, top_features_idx[:100]]

# Make predictions with the random forest model using the selected features
y_pred_new = forest_model_top_500_features.predict(X_new_top_100)

# Extract true labels from the new dataset
y_true_new = new_data["metastasis_status"]

# Calculate accuracy
accuracy_new = accuracy_score(y_true_new, y_pred_new)

# Display accuracy
print("Accuracy on the new dataset:", accuracy_new)

# Display confusion matrix
conf_matrix = confusion_matrix(y_true_new, y_pred_new)
print("\nConfusion Matrix:")
print(conf_matrix)


Model 2

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LassoCV
import joblib

# Assuming 'metastasis_status' is the target variable
target_variable = 'metastasis_status'

# Extract 'cn' features and target variable
X_cn = df[[col for col in df.columns if col.startswith('cn')]]
target = df[target_variable]

# Apply SMOTE to the 'cn' features only
smote = SMOTE(sampling_strategy=0.6, random_state=42)
X_cn_augmented, y_cn_augmented = smote.fit_resample(X_cn, target)

# Split the augmented data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_cn_augmented, y_cn_augmented, test_size=0.2, random_state=42)

# Create a LASSO model using cross-validation
lasso_model = LassoCV(cv=5, max_iter=10000, random_state=42)

# Fit the LASSO model
lasso_model.fit(X_train, y_train)

# Select features based on non-zero coefficients
selected_features = X_train.columns[lasso_model.coef_ != 0]

# Subset the data with selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Create a random forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the selected features
rf_classifier.fit(X_train_selected, y_train)

# Make predictions on the test set
predictions = rf_classifier.predict(X_test_selected)

# Evaluate the model
accuracy = accuracy_score(y_test, predictions)
conf_matrix = confusion_matrix(y_test, predictions)
class_report = classification_report(y_test, predictions)

# Display the results
print("Random Forest Classifier with SMOTE and LASSO feature selection:")
print(f"Accuracy: {accuracy:.2f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(class_report)


In [ ]:
from sklearn.metrics import cohen_kappa_score

# Calculate Cohen's Kappa
kappa_cn = cohen_kappa_score(y_test, predictions)

# Display Cohen's Kappa
print("\nCohen's Kappa for Model 2 with SMOTE and PCA before data split:")
print(kappa_cn)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get predicted probabilities for the positive class
y_prob_rf = rf_classifier.predict_proba(X_test_pca)[:, 1]

# Calculate ROC curve
fpr_cn, tpr_cn, thresholds_cn = roc_curve(y_test, y_prob_rf)

# Calculate AUC (Area Under the Curve)
roc_auc_cn = auc(fpr_cn, tpr_cn)
# Store ROC curve data in variables
final_roc_curve_cn = {
    'fpr': fpr_cn,
    'tpr': tpr_cn,
    'thresholds': thresholds_cn,
    'roc_auc': roc_auc_cn
}
# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_cn, tpr_cn, color='darkorange', lw=2, label='ROC curve (AUC = {:.2f})'.format(roc_auc_cn))
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()


In [ ]:
# Save the RandomForestClassifier model
joblib.dump(rf_classifier, "random_forest_model.joblib")

# Save the PCA model for consistency in feature transformation
joblib.dump(pca, "pca_model.joblib")


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score
import joblib

# Load the saved PCA model
loaded_pca = joblib.load("pca_model.joblib")

# Load the trained RandomForestClassifier model
loaded_rf_model = joblib.load("random_forest_model.joblib")

# Load the new dataset for prediction
new_data_nsclc = pd.read_csv("/content/NSCLCMultiOmicsDataProcessed_reordered.csv")

# Extract the features (cn_ data) from the new dataset
X_new = new_data_nsclc[[col for col in new_data_nsclc.columns if col.startswith('cn')]]

# Apply PCA to the new data without relying on original feature names
X_pca_cn = loaded_pca.transform(X_new)

# Make predictions on the new dataset using the loaded RandomForestClassifier model
y_pred_new = loaded_rf_model.predict(X_pca_cn)

# Extract true labels from the new dataset
y_true_new = new_data_nsclc["metastasis_status"]

# Calculate accuracy
accuracy_new = accuracy_score(y_true_new, y_pred_new)

# Display accuracy
print("Accuracy on the new dataset without SMOTE:", accuracy_new)

# Display predictions
predictions_df = pd.DataFrame({
    "True Labels": y_true_new,
    "Predicted Labels": y_pred_new
})

print("Predictions on the new dataset without SMOTE:")
print(predictions_df)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LassoCV
import joblib

# Assuming 'metastasis_status' is the target variable
target_variable = 'metastasis_status'

# Extract features (mu_ data) and target variable for df_mu
X_mu = df[[col for col in df.columns if col.startswith('mu_')]]
target_mu = df[target_variable]

# Apply SMOTE to the mu_ features only
smote = SMOTE(sampling_strategy="auto", random_state=42)
X_mu_augmented, y_mu_augmented = smote.fit_resample(X_mu, target_mu)

# Split the augmented data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_mu_augmented, y_mu_augmented, test_size=0.2, random_state=42)

# Create a LASSO model using cross-validation
lasso_model = LassoCV(cv=5, max_iter=10000, random_state=42)

# Fit the LASSO model
lasso_model.fit(X_train, y_train)

# Select features based on non-zero coefficients
selected_features = X_train.columns[lasso_model.coef_ != 0]

# Subset the data with selected features
X_train_selected = X_train[selected_features]
X_test_selected_mu = X_test[selected_features]

# Create a random forest classifier
rf_classifier_mu = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the reduced features
rf_classifier_mu.fit(X_train_selected, y_train)

# Make predictions on the test set
predictions_mu = rf_classifier_mu.predict(X_test_selected_mu)

# Evaluate the model
accuracy_mu = accuracy_score(y_test, predictions_mu)
conf_matrix_mu = confusion_matrix(y_test, predictions_mu)
class_report_mu = classification_report(y_test, predictions_mu)

# Display the results
print("Random Forest Classifier with LASSO and SMOTE for df_mu:")
print(f"Accuracy: {accuracy_mu:.2f}")
print("\nConfusion Matrix:")
print(conf_matrix_mu)
print("\nClassification Report:")
print(class_report_mu)


In [ ]:
from sklearn.metrics import cohen_kappa_score

# Calculate Cohen's Kappa
kappa_mu = cohen_kappa_score(y_test_mu, predictions_mu)

# Display Cohen's Kappa
print("\nCohen's Kappa for df_mu with Autoencoder and SMOTE:")
print(kappa_mu)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get predicted probabilities for the positive class
y_prob_rf_mu = rf_classifier_mu.predict_proba(X_test_pca)[:, 1]

# Calculate ROC curve
fpr_mu, tpr_mu, thresholds_mu = roc_curve(y_test_mu, y_prob_rf_mu)

# Calculate AUC (Area Under the Curve)
roc_auc_mu = auc(fpr_mu, tpr_mu)
# Store ROC curve data in variables
final_roc_curve_mu = {
    'fpr': fpr_mu,
    'tpr': tpr_mu,
    'thresholds': thresholds_mu,
    'roc_auc': roc_auc_mu
}
# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_mu, tpr_mu, color='darkorange', lw=2, label='ROC curve (AUC = {:.2f})'.format(roc_auc_mu))
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve for df_mu')
plt.legend(loc='lower right')
plt.show()


In [ ]:
import joblib

# Save the trained PCA model to a file
pca_model_filename = 'pca_model_mu.joblib'
joblib.dump(pca, pca_model_filename)

# Save the trained Random Forest Classifier model to a file
rf_model_filename = 'random_forest_model_mu.joblib'
joblib.dump(rf_classifier_mu, rf_model_filename)

print(f"PCA model saved to {pca_model_filename}")
print(f"Random Forest model saved to {rf_model_filename}")


In [ ]:
#Model 3

import pandas as pd
from sklearn.metrics import accuracy_score
import joblib

# Load the saved PCA model
loaded_pca = joblib.load("pca_model_mu.joblib")

# Load the trained RandomForestClassifier model
loaded_rf_model = joblib.load("random_forest_model_mu.joblib")

# Load the new dataset for prediction
new_data_nsclc = pd.read_csv("/content/NSCLCMultiOmicsDataProcessed_reordered.csv")

# Extract the features (cn_ data) from the new dataset
X_new = new_data_nsclc[[col for col in new_data_nsclc.columns if col.startswith('mu')]]

# Apply PCA to the new data using the original feature names
X_new_pca_mu = loaded_pca.transform(X_new)

# Make predictions on the new dataset using the loaded RandomForestClassifier model
y_pred_new = loaded_rf_model.predict(X_new_pca_mu)

# Extract true labels from the new dataset
y_true_new = new_data_nsclc["metastasis_status"]

# Calculate accuracy
accuracy_new = accuracy_score(y_true_new, y_pred_new)

# Display accuracy
print("Accuracy on the new dataset:", accuracy_new)


# Display predictions
predictions_df = pd.DataFrame({
    "True Labels": y_true_new,
    "Predicted Labels": y_pred_new
})

print("Predictions on the new dataset:")
print(predictions_df)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LassoCV
import joblib

# Assuming 'metastasis_status' is the target variable
target_variable = 'metastasis_status'

# Extract features (pp_ data) and target variable for df_pp
X_pp = df[[col for col in df.columns if col.startswith('pp')]]
target_pp = df[target_variable]

# Apply SMOTE to the entire dataset
smote = SMOTE(sampling_strategy=0.3, random_state=42)
X_augmented_pp, y_augmented_pp = smote.fit_resample(X_pp, target_pp)

# Split the augmented data into training and testing sets
X_train_pp, X_test_pp, y_train_pp, y_test_pp = train_test_split(X_augmented_pp, y_augmented_pp, test_size=0.2, random_state=42)

# Create a LASSO model using cross-validation
lasso_model = LassoCV(cv=5, max_iter=10000, random_state=42)

# Fit the LASSO model
lasso_model.fit(X_train_pp, y_train_pp)

# Select features based on non-zero coefficients
selected_features = X_train_pp.columns[lasso_model.coef_ != 0]

# Subset the data with selected features
X_train_selected_pp = X_train_pp[selected_features]
X_test_selected_pp = X_test_pp[selected_features]

# Create a random forest classifier
rf_classifier_pp = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the reduced features
rf_classifier_pp.fit(X_train_selected_pp, y_train_pp)

# Make predictions on the test set
predictions_pp = rf_classifier_pp.predict(X_test_selected_pp)

# Evaluate the model
accuracy_pp = accuracy_score(y_test_pp, predictions_pp)
conf_matrix_pp = confusion_matrix(y_test_pp, predictions_pp)
class_report_pp = classification_report(y_test_pp, predictions_pp)

# Display the results for df_pp with LASSO and SMOTE
print("Random Forest Classifier with LASSO and SMOTE for df_pp:")
print(f"Accuracy: {accuracy_pp:.2f}")
print("\nConfusion Matrix:")
print(conf_matrix_pp)
print("\nClassification Report:")
print(class_report_pp)


In [ ]:
# Get feature importances from the trained random forest model
feature_importances_pp = rf_classifier_pp.feature_importances_

# Get the indices of the top 30 features
top_30_proteins_idx_pp = feature_importances_pp.argsort()[-30:][::-1]

# Get the corresponding protein names without the 'pp' prefix
top_30_proteins_pp = X_train_pp.columns[top_30_proteins_idx_pp]
top_30_proteins_without_prefix_pp = [col[3:] for col in top_30_proteins_pp]

# Display the top 30 proteins most associated with NSCLC metastasis
print("\nTop 30 Proteins Most Associated with NSCLC Metastasis:")
for i, protein in enumerate(top_30_proteins_without_prefix_pp, 1):
    print(f"{i}. {protein}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Get the top 30 proteins and their corresponding importances
top_30_proteins_importance_pp = zip(top_30_proteins_without_prefix_pp, feature_importances_pp[top_30_proteins_idx_pp])

# Create a DataFrame for visualization
proteins_importance_df_pp = pd.DataFrame(top_30_proteins_importance_pp, columns=['Protein', 'Importance'])

# Transpose the DataFrame for better visualization
proteins_importance_transposed_df_pp = proteins_importance_df_pp

# Create a heatmap
plt.figure(figsize=(12, 8))
heatmap = sns.heatmap(proteins_importance_transposed_df_pp[['Importance']], cmap='viridis', cbar_kws={'label': 'Protein Expression'}, xticklabels=False, annot=False)

# Label the y-axis with protein names
heatmap.set_yticklabels(proteins_importance_transposed_df_pp['Protein'], rotation=0)

plt.title('Top 30 Proteins Associated with NSCLC Metastasis')
plt.xlabel('Samples (Not Labeled)')
plt.ylabel('Proteins')

# Show the plot
plt.show()


In [ ]:
from sklearn.metrics import cohen_kappa_score

# Calculate Cohen's Kappa
kappa_pp = cohen_kappa_score(y_test_pp, predictions_pp)

# Display Cohen's Kappa
print("\nCohen's Kappa for df_pp with Autoencoder and SMOTE:")
print(kappa_pp)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get predicted probabilities for the positive class
y_prob_rf_pp = rf_classifier_pp.predict_proba(X_test_pca)[:, 1]

# Calculate ROC curve
fpr_pp, tpr_pp, thresholds_pp = roc_curve(y_test_pp, y_prob_rf_pp)

# Calculate AUC (Area Under the Curve)
roc_auc_pp = auc(fpr_pp, tpr_pp)
# Store ROC curve data in variables
final_roc_curve_pp = {
    'fpr': fpr_pp,
    'tpr': tpr_pp,
    'thresholds': thresholds_pp,
    'roc_auc': roc_auc_pp
}
# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr_pp, tpr_pp, color='darkorange', lw=2, label='ROC curve (AUC = {:.2f})'.format(roc_auc_pp))
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve for df_pp')
plt.legend(loc='lower right')
plt.show()


In [ ]:
import joblib

# ... (previous code remains unchanged)

# Save the trained PCA model to a file
pca_filename = 'pca_model_pp.joblib'
joblib.dump(pca, pca_filename)

# Save the trained Random Forest Classifier model to a file
model_filename = 'random_forest_model_pp.joblib'
joblib.dump(rf_classifier_pp, model_filename)

print(f"PCA model saved to {pca_filename}")
print(f"Random Forest Classifier model saved to {model_filename}")


In [ ]:
#Model 4

import pandas as pd
from sklearn.metrics import accuracy_score
import joblib

# Load the saved PCA model
loaded_pca = joblib.load("pca_model_pp.joblib")

# Load the trained RandomForestClassifier model
loaded_rf_model = joblib.load("random_forest_model_pp.joblib")

# Load the new dataset for prediction
new_data_nsclc = pd.read_csv("/content/NSCLCMultiOmicsDataProcessed_reordered.csv")

# Extract the features (cn_ data) from the new dataset
X_new = new_data_nsclc[[col for col in new_data_nsclc.columns if col.startswith('pp')]]

# Apply PCA to the new data using the original feature names
X_new_pca_pp = loaded_pca.transform(X_new)

# Make predictions on the new dataset using the loaded RandomForestClassifier model
y_pred_new = loaded_rf_model.predict(X_new_pca_pp)

# Extract true labels from the new dataset
y_true_new = new_data_nsclc["metastasis_status"]

# Calculate accuracy
accuracy_new = accuracy_score(y_true_new, y_pred_new)

# Display accuracy
print("Accuracy on the new dataset:", accuracy_new)


# Display predictions
predictions_df = pd.DataFrame({
    "True Labels": y_true_new,
    "Predicted Labels": y_pred_new
})

print("Predictions on the new dataset:")
print(predictions_df)

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

# Load the new dataset
new_data = pd.read_csv("/content/NSCLCMultiOmicsDataProcessed_reordered.csv")

# Extract the true labels from the new dataset
y_true_new = new_data["metastasis_status"]

# Load the saved models
loaded_rf_model_top_500_features = joblib.load("random_forest_model_top_500_features.joblib")
loaded_rf_model_cn = joblib.load("random_forest_model.joblib")
loaded_rf_model_mu = joblib.load("random_forest_model_mu.joblib")
loaded_rf_model_pp = joblib.load("random_forest_model_pp.joblib")



In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Load and preprocess datasets (assuming it's done elsewhere in the code)

# Initialize an empty DataFrame to store predictions
predictions_df = pd.DataFrame(index=X_train.index)

# Specify the number of folds for cross-validation
n_splits = 5

# Initialize lists to store evaluation metrics for each fold
accuracy_scores = []
kappa_scores = []
conf_matrices = []

# Initialize StratifiedKFold object for cross-validation
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Perform cross-validation
for fold, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
    X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    # Use RandomForest as the final estimator for the Stacking Classifier
    final_estimator = RandomForestClassifier(n_estimators=100, random_state=42)

    # Create the Stacking Classifier
    stacking_classifier = StackingClassifier(
        estimators=[
            ('RF_Model_Top_500_Features', loaded_rf_model_top_500_features),
            ('RF_Model_CN', loaded_rf_model_cn),
            ('RF_Model_MU', loaded_rf_model_mu),
            ('RF_Model_PP', loaded_rf_model_pp)
        ],
        final_estimator=final_estimator,
        stack_method='auto',
        passthrough=True
    )

    # Fit the Stacking Classifier on the training data
    stacking_classifier.fit(X_train_fold, y_train_fold)

    # Predict on the validation data
    y_pred_val = stacking_classifier.predict(X_val_fold)

    # Add predictions to the DataFrame
    predictions_df[f'Fold_{fold+1}_Predictions'] = np.nan
    predictions_df.loc[X_val_fold.index, f'Fold_{fold+1}_Predictions'] = y_pred_val

    # Evaluate performance on the validation set
    accuracy_fold = accuracy_score(y_val_fold, y_pred_val)
    kappa_fold = cohen_kappa_score(y_val_fold, y_pred_val)
    conf_matrix_fold = confusion_matrix(y_val_fold, y_pred_val)

    # Store evaluation metrics for this fold
    accuracy_scores.append(accuracy_fold)
    kappa_scores.append(kappa_fold)
    conf_matrices.append(conf_matrix_fold)

# Calculate mean and standard deviation of evaluation metrics across all folds
mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)
mean_kappa = np.mean(kappa_scores)
std_kappa = np.std(kappa_scores)

# Display mean and standard deviation of evaluation metrics
print("Mean Accuracy:", mean_accuracy)
print("Standard Deviation of Accuracy:", std_accuracy)
print("Mean Cohen's Kappa:", mean_kappa)
print("Standard Deviation of Cohen's Kappa:", std_kappa)

# Display mean confusion matrix across all folds
mean_conf_matrix = np.mean(conf_matrices, axis=0)
print("\nMean Confusion Matrix (across all folds):")
print(mean_conf_matrix)


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_predict, train_test_split
import joblib

# Load and process your datasets
# ...

# Assuming PCA models and feature extraction
# ...

# Make predictions using each model
y_pred_rf_top_500_features = loaded_rf_model_top_500_features.predict(X_test_std[:, top_features_idx])
y_pred_rf_cn = loaded_rf_model_cn.predict(X_test_selected)
y_pred_rf_mu = loaded_rf_model_mu.predict(X_test_selected_mu)
y_pred_rf_pp = loaded_rf_model_pp.predict(X_test_selected_pp)

# Create a DataFrame with the predictions from each model
predictions_df = pd.DataFrame({
    "RF_Model_Top_500_Features": y_pred_rf_top_500_features,
    "RF_Model_CN": y_pred_rf_cn,
    "RF_Model_MU": y_pred_rf_mu,
    "RF_Model_PP": y_pred_rf_pp
})

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(predictions_df, y_true_new, test_size=0.2, random_state=42)

# Use RandomForest as the final estimator for the Stacking Classifier
final_estimator = RandomForestClassifier(n_estimators=100, random_state=42)

# Create the Stacking Classifier
stacking_classifier = StackingClassifier(
    estimators=[
        ('RF_Model_Top_500_Features', loaded_rf_model_top_500_features),
        ('RF_Model_CN', loaded_rf_model_cn),
        ('RF_Model_MU', loaded_rf_model_mu),
        ('RF_Model_PP', loaded_rf_model_pp)
    ],
    final_estimator=final_estimator,
    stack_method='auto',
    passthrough=True,
    cv=5
)

# Fit and predict using cross_val_predict
y_pred_stacking = cross_val_predict(stacking_classifier, X_train, y_train, cv=5)

# Calculate accuracy
accuracy_stacking = accuracy_score(y_train, y_pred_stacking)

# Display accuracy
print("Accuracy with Stacking Classifier:", accuracy_stacking)

# Display confusion matrix
conf_matrix_stacking = confusion_matrix(y_train, y_pred_stacking)
print("\nConfusion Matrix (Stacking Classifier):")
print(conf_matrix_stacking)

# Calculate Cohen's Kappa
kappa_stacking = cohen_kappa_score(y_train, y_pred_stacking)

# Display Cohen's Kappa
print("\nCohen's Kappa (Stacking Classifier):", kappa_stacking)

# Print out every dataset
print("\nPredictions from each model:")
print(predictions_df)

print(y_pred_stacking)


In [ ]:
from scipy.stats import chi2_contingency

# Create a contingency table
contingency_table = pd.crosstab(y_train, y_pred_stacking)

# Perform the Chi-squared test
chi2_stat, p_value, _, _ = chi2_contingency(contingency_table)

# Display the results
print("Chi-squared Statistic:", chi2_stat)
print("P-value:", p_value)

# Check for significance
alpha = 0.05
if p_value < alpha:
    print("The result is statistically significant - reject the null hypothesis.")
else:
    print("The result is not statistically significant - fail to reject the null hypothesis.")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot the contingency table as a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(contingency_table, annot=True, cmap="YlGnBu", fmt="d", cbar=False)
plt.title("Confusion Matrix (Stacking Classifier)")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()


In [ ]:
# Calculate ROC curve for each class
fpr_fs, tpr_fs, thresholds_fs = roc_curve(y_train, y_pred_stacking)  # Assuming binary classification, adjust accordingly

# Calculate AUC (Area Under the Curve)
roc_auc_fs = auc(fpr_fs, tpr_fs)


final_roc_curve_fs = {
    'fpr': fpr_fs,
    'tpr': tpr_fs,
    'thresholds': thresholds_fs,
    'roc_auc': roc_auc_fs
}
# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr_fs, tpr_fs, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_fs:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Stacking Classifier')
plt.legend(loc='lower right')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Assuming you have dictionaries with ROC curve data for each model
# Replace these dictionaries with your actual data

final_roc_curve_fs = {
    'fpr': fpr_fs,
    'tpr': tpr_fs,
    'thresholds': thresholds_fs,
    'roc_auc': roc_auc_fs
}

final_roc_curve_pp = {
    'fpr': fpr_pp,
    'tpr': tpr_pp,
    'thresholds': thresholds_pp,
    'roc_auc': roc_auc_pp
}

final_roc_curve_mu = {
    'fpr': fpr_mu,
    'tpr': tpr_mu,
    'thresholds': thresholds_mu,
    'roc_auc': roc_auc_mu
}

final_roc_curve_cn = {
    'fpr': fpr_cn,
    'tpr': tpr_cn,
    'thresholds': thresholds_cn,
    'roc_auc': roc_auc_cn
}

final_roc_curve_rs = {
    'fpr': fpr_rs,
    'tpr': tpr_rs,
    'thresholds': thresholds_rs,
    'roc_auc': roc_auc_rs
}

# Plotting ROC Curves
plt.figure(figsize=(10, 8))

plt.plot(final_roc_curve_fs['fpr'], final_roc_curve_fs['tpr'], color='darkorange', lw=2, label='Multi-Model Ensemble - AUC = {:.2f}'.format(final_roc_curve_fs['roc_auc']))
plt.plot(final_roc_curve_pp['fpr'], final_roc_curve_pp['tpr'], color='green', lw=2, label='RPPA Model - AUC = {:.2f}'.format(final_roc_curve_pp['roc_auc']))
plt.plot(final_roc_curve_mu['fpr'], final_roc_curve_mu['tpr'], color='blue', lw=2, label='MU Model - AUC = {:.2f}'.format(final_roc_curve_mu['roc_auc']))
plt.plot(final_roc_curve_cn['fpr'], final_roc_curve_cn['tpr'], color='purple', lw=2, label='CNV Model - AUC = {:.2f}'.format(final_roc_curve_cn['roc_auc']))
plt.plot(final_roc_curve_rs['fpr'], final_roc_curve_rs['tpr'], color='red', lw=2, label='RS Model - AUC = {:.2f}'.format(final_roc_curve_rs['roc_auc']))

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Chance')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.show()


In [ ]:
print(df_cn)

In [ ]:
print(df_rs)

In [ ]:
print(df_mu)

In [ ]:
print(df_pp)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Replace these dictionaries with your actual accuracy data
model_accuracies = {
    'Multi-Model Ensemble': accuracy_stacking * 100,
    'RPPA Model': accuracy_pp * 100,
    'MU Model': accuracy_mu * 100,
    'CNV Model': accuracy * 100,
    'RS Model': accuracy_forest_top_500_features * 100
}

# Bar graph of all model accuracies
models = list(model_accuracies.keys())
accuracies = list(model_accuracies.values())

plt.figure(figsize=(10, 6))
plt.bar(models, accuracies, color=['darkorange', 'green', 'blue', 'purple', 'red'])
plt.xlabel('Models')
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracies')
plt.ylim(50, 100)  # Assuming accuracy is in the range [0, 100]
plt.show()

# Bar graph with 85% accuracy line
average_accuracy = np.mean(accuracies)

plt.figure(figsize=(10, 6))
plt.bar(models, accuracies, color=['darkorange', 'green', 'blue', 'purple', 'red'])
plt.axhline(y=85, color='r', linestyle='--', label= ' Average Model Accuracy of Previous Approaches (85%)')
plt.xlabel('Models')
plt.ylabel('Accuracy (%)')
plt.title('Accuracies for All Models Developed')
plt.ylim(50, 100)  # Assuming accuracy is in the range [0, 100]
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Cohen's Kappa values for each model
kappa_values = [
    kappa_forest_top_500_features,
    kappa_cn,
    kappa_mu,
    kappa_pp,
    kappa_stacking
]

# Benchmark kappa value
benchmark_kappa = 0.81

# Model names
model_names = [
    "RS Model",
    "CNV Model",
    "MU Model",
    "RPPA Model",
    "Multi-Model Ensemble"
]

# Create a larger bar graph
plt.figure(figsize=(12, 8))
bars = plt.bar(model_names, kappa_values, color=['blue', 'green', 'orange', 'purple', 'red'])
plt.xlabel('Models')
plt.ylabel("Cohen's Kappa")
plt.title('Cohen\'s Kappa Values for Different Models')
plt.ylim([-1, 1])  # Set y-axis limits to -1 to 1 for consistency

# Display the Cohen's Kappa values at the center of the bars
for bar, value in zip(bars, kappa_values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, f'{value:.2f}', ha='center', va='center', color='white')

# Add a horizontal line for the benchmark kappa value
plt.axhline(y=benchmark_kappa, color='black', linestyle='--', label='Kappa result interpreted as almost perfect agreement')

# Show the legend
plt.legend()

# Show the plot
plt.show()
